In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/HHS_Unaccompanied_Alien_Children_Program.csv")
df.columns = ["date", "apprehended", "cbp_custody",
              "cbp_transferred", "hhs_care", "hhs_discharged"]
df = df.dropna(how="all").copy()
df["hhs_care"] = df["hhs_care"].str.replace(",", "").astype(int)
df["date"] = pd.to_datetime(df["date"], format="%B %d, %Y")
df = df.sort_values("date").reset_index(drop=True)

num_cols = ["apprehended", "cbp_custody", "cbp_transferred",
            "hhs_care", "hhs_discharged"]
print(df.shape)     # (720, 6)

(720, 6)


In [2]:
print("negatives:", (df[num_cols] < 0).sum().sum())
print("decimals :", (df[num_cols] % 1 != 0).sum().sum())
print("missing  :", df[num_cols].isna().sum().sum())

negatives: 0
decimals : 0
missing  : 0


In [3]:
df["transfer_gt_custody"] = df["cbp_transferred"] > df["cbp_custody"]
df["discharge_gt_hhs"] = df["hhs_discharged"] > df["hhs_care"]
print(df[["transfer_gt_custody", "discharge_gt_hhs"]].sum())

# How big are the violations?
df["transfer_excess"] = (df["cbp_transferred"] - df["cbp_custody"]).clip(lower=0)
print(df.loc[df["transfer_gt_custody"], "transfer_excess"].describe())

# Which years?
df.groupby(df["date"].dt.year)["transfer_gt_custody"].agg(["sum", "count"])

transfer_gt_custody    86
discharge_gt_hhs        0
dtype: int64
count    86.000000
mean     23.581395
std      17.613140
min       1.000000
25%      10.250000
50%      18.000000
75%      32.500000
max      73.000000
Name: transfer_excess, dtype: float64


,sum,count
date,,
2023,43,230
2024,31,251
2025,12,239


In [4]:
flow_cols = ["apprehended", "cbp_transferred", "hhs_discharged"]
df["zero_flag"] = (df[flow_cols] == 0).any(axis=1)
print(df["zero_flag"].sum())      # 6
df.loc[df["zero_flag"], ["date"] + flow_cols]

6


,date,apprehended,cbp_transferred,hhs_discharged
533,2025-03-17,5.0,0.0,8.0
607,2025-07-09,2.0,0.0,32.0
611,2025-07-15,0.0,5.0,17.0
612,2025-07-16,1.0,0.0,11.0
628,2025-08-10,0.0,8.0,12.0
704,2025-11-30,4.0,10.0,0.0


In [5]:
df["gap_days"] = df["date"].diff().dt.days
df["hhs_change"] = df["hhs_care"].diff()
df["hhs_pct_per_day"] = df["hhs_care"].pct_change() * 100 / df["gap_days"]

df["hhs_pct_per_day"].abs().describe()
print((df["hhs_pct_per_day"].abs() > 3).sum())     # 14

14


In [6]:
cons = df[df["gap_days"] == 1].copy()          # consecutive days only
cons["expected_change"] = cons["cbp_transferred"] - cons["hhs_discharged"]
cons["difference"] = cons["hhs_change"] - cons["expected_change"]

print(len(cons))                                    # 558
print((cons["difference"].abs() <= 20).mean())      # about 0.52
cons.groupby(cons["date"].dt.year)["difference"].median()

558
0.5232974910394266


date
2023    192.5
2024      6.0
2025      3.0
Name: difference, dtype: float64

In [7]:
parts = []

r1 = df[df["transfer_gt_custody"]]
parts.append(pd.DataFrame({
    "date": r1["date"], "rule": "R1_transfers_exceed_custody", "severity": "warning",
    "details": "transferred=" + r1["cbp_transferred"].astype(int).astype(str)
               + ", custody=" + r1["cbp_custody"].astype(int).astype(str)}))

r2 = df[df["discharge_gt_hhs"]]
parts.append(pd.DataFrame({
    "date": r2["date"], "rule": "R2_discharges_exceed_hhs_care", "severity": "error",
    "details": "discharged=" + r2["hhs_discharged"].astype(int).astype(str)
               + ", hhs_care=" + r2["hhs_care"].astype(str)}))

r3 = df[df["zero_flag"]]
parts.append(pd.DataFrame({
    "date": r3["date"], "rule": "R3_zero_flow_value", "severity": "info",
    "details": "apprehended=" + r3["apprehended"].astype(int).astype(str)
               + ", transferred=" + r3["cbp_transferred"].astype(int).astype(str)
               + ", discharged=" + r3["hhs_discharged"].astype(int).astype(str)}))

r4 = df[df["hhs_pct_per_day"].abs() > 3]
parts.append(pd.DataFrame({
    "date": r4["date"], "rule": "R4_large_hhs_change", "severity": "review",
    "details": r4["hhs_pct_per_day"].round(2).astype(str) + "% per day over "
               + r4["gap_days"].astype(int).astype(str) + " day(s)"}))

anomalies = (pd.concat(parts, ignore_index=True)
               .sort_values(["date", "rule"]).reset_index(drop=True))

print(anomalies.shape)
print(anomalies.groupby(["rule", "severity"]).size())
print("dates with any flag:", anomalies["date"].nunique(), "of", len(df))

anomalies.to_csv("../data/processed/anomaly_report.csv", index=False)

(106, 4)
rule                         severity
R1_transfers_exceed_custody  warning     86
R3_zero_flow_value           info         6
R4_large_hhs_change          review      14
dtype: int64
dates with any flag: 104 of 720
